In [ ]:
#@title 01  ·  Install { display-mode: "form" }
import os, re, sys, time, shutil, subprocess, html
from IPython.display import display, HTML

COMFY = "/content/ComfyUI"
VERSION = "v0.39.2"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"

CSS = r"""
<style>
.om-wrap{width:98%;margin:0 auto;box-sizing:border-box;background:#141210;color:#e8dfd0;
border:1px solid #3f382c;padding:28px 32px 22px;font-family:Palatino,"Palatino Linotype","Book Antiqua",Georgia,serif;}
.om-top{display:flex;justify-content:space-between;align-items:baseline;gap:16px;}
.om-kicker{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.28em;color:#b8956c;text-transform:uppercase;}
.om-status{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.22em;text-transform:uppercase;}
.om-s-WORKING{color:#b8956c;} .om-s-READY,.om-s-LIVE{color:#8fa382;} .om-s-ERROR{color:#8a3a3a;} .om-s-STOPPED{color:#8a8070;}
.om-title{font-size:28px;font-weight:500;letter-spacing:.04em;color:#f3eadb;margin:6px 0 0;line-height:1.15;}
.om-rule{height:1px;background:#3f382c;margin:16px 0 18px;border:0;}
.om-row{display:flex;gap:18px;justify-content:space-between;padding:5px 0;font-size:16px;line-height:1.4;}
.om-k{color:#8a8070;min-width:160px;} .om-v{color:#e8dfd0;text-align:right;}
.om-meta{font-family:"Courier New",Courier,monospace;font-size:11px;color:#a89880;letter-spacing:.04em;}
.om-foot{margin-top:16px;font-family:"Courier New",Courier,monospace;font-size:11px;color:#8a8070;letter-spacing:.06em;text-transform:uppercase;}
.om-log{background:#0e0c0a;border:1px solid #2a261f;padding:12px 14px;max-height:280px;overflow:auto;
font-family:"Courier New",Courier,monospace;font-size:11px;line-height:1.55;color:#c8bba6;white-space:pre-wrap;word-break:break-word;}
.om-bar-track{height:2px;background:#2a261f;margin:14px 0 8px;} .om-bar-fill{height:2px;background:#b8956c;width:0;}
.om-note{color:#a89880;font-size:15px;line-height:1.5;}
</style>
"""

def esc(s):
    return html.escape(str(s) if s is not None else "")

class OMBox:
    def __init__(self, no, title):
        self.no, self.title = no, title
        self.status, self.rows, self.foot, self.extra = "WORKING", [], "", ""
        self._h = None
        self.paint()
    def paint(self):
        rows = "".join(
            f'<div class="om-row"><span class="om-k">{esc(k)}</span><span class="om-v">{esc(v)}</span></div>'
            for k, v in self.rows
        )
        h = HTML(CSS + f"""
        <div class="om-wrap">
          <div class="om-top"><div class="om-kicker">{esc(self.no)}</div>
            <div class="om-status om-s-{esc(self.status)}">{esc(self.status)}</div></div>
          <div class="om-title">{esc(self.title)}</div>
          <div class="om-rule"></div>
          {rows}{self.extra}
          <div class="om-foot">{esc(self.foot)}</div>
        </div>""")
        if self._h is None:
            self._h = display(h, display_id=f"om-{self.no}")
        else:
            self._h.update(h)
    def set(self, rows=None, status=None, foot=None, extra=None):
        if rows is not None: self.rows = rows
        if status is not None: self.status = status
        if foot is not None: self.foot = foot
        if extra is not None: self.extra = extra
        self.paint()

ui = OMBox("01", "Installation")
t0 = time.time()

def run(cmd, cwd=None, timeout=None):
    p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    return p.returncode, p.stdout or "", p.stderr or ""

def elapsed():
    s = int(time.time() - t0)
    return f"{s//60}m {s%60:02d}s" if s >= 60 else f"{s}s"

rows = []
def tick(key, val):
    found = False
    for i, (k, _) in enumerate(rows):
        if k == key:
            rows[i] = (key, val); found = True; break
    if not found:
        rows.append((key, val))
    ui.set(rows=rows, foot=f"running  ·  {elapsed()}")

# --- gpu ---
tick("runtime", "detecting device")
gpu, vram, cuda = "none", "—", "—"
try:
    c, o, _ = run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits")
    if c == 0 and o.strip():
        name, mem = [x.strip() for x in o.strip().splitlines()[0].split(",")]
        gpu, vram = name, f"{mem} MB"
    import torch
    cuda = f"{torch.version.cuda}" if torch.cuda.is_available() else "cpu"
    if not torch.cuda.is_available():
        ui.set(status="ERROR", foot="no gpu  ·  runtime → change runtime type → gpu")
except Exception as e:
    tick("runtime", str(e))
tick("device", f"{gpu}  ·  cuda {cuda}  ·  {vram}")

# --- system ---
tick("system", "apt packages")
run("apt-get update -qq")
run("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq git wget curl ffmpeg aria2 libgl1 libglib2.0-0 >/dev/null")
tick("system", "ready")

tick("tunnel binary", "cloudflared")
if not os.path.isfile("/usr/local/bin/cloudflared"):
    run("wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    run("chmod +x /usr/local/bin/cloudflared")
tick("tunnel binary", "cloudflared ready")

# --- comfyui ---
tick("comfyui", f"fetching {VERSION}")
ver_used = VERSION
if os.path.isdir(os.path.join(COMFY, ".git")):
    run("git fetch --tags --force", cwd=COMFY)
    for cand in (VERSION, VERSION.lstrip("v"), f"v{VERSION.lstrip('v')}"):
        if run(f"git checkout {cand}", cwd=COMFY)[0] == 0:
            ver_used = cand; break
    else:
        run("git checkout master", cwd=COMFY); ver_used = "latest"
else:
    if os.path.isdir(COMFY):
        shutil.rmtree(COMFY)
    c, _, e = run(f"git clone --branch {VERSION} --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY}")
    if c != 0:
        run(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY}")
        ver_used = "latest"
tick("comfyui", f"{ver_used}  ·  {COMFY}")

# --- python deps (keep the colab torch build) ---
tick("dependencies", "installing")
req = os.path.join(COMFY, "requirements.txt")
filtered = "/tmp/comfy_req_notorch.txt"
if os.path.isfile(req):
    with open(req) as f, open(filtered, "w") as g:
        for ln in f:
            if not re.match(r"\s*(torch|torchvision|torchaudio)\b", ln, re.I):
                g.write(ln)
    run(f"{sys.executable} -m pip install -q -U pip setuptools wheel")
    run(f"{sys.executable} -m pip install -q -r {filtered}")
else:
    run(f"{sys.executable} -m pip install -q -U pip setuptools wheel")

# kitchen backend: official frontend split + server extras
tick("kitchen backend", "frontend · templates · server extras")
run(f"{sys.executable} -m pip install -q -U huggingface_hub hf_transfer requests packaging "
    "comfyui-frontend-package comfyui-workflow-templates comfyui-embedded-docs "
    "GitPython rich typer matrix-client transformers accelerate safetensors "
    "einops scipy pillow pyyaml tqdm psutil aiohttp yarl alembic SQLAlchemy av soundfile")
try:
    run(f"{sys.executable} -m pip install -q xformers")
except Exception:
    pass
tick("kitchen backend", "ready")
tick("dependencies", "ready")

# --- manager ---
mgr = os.path.join(COMFY, "custom_nodes", "ComfyUI-Manager")
tick("manager", "installing")
os.makedirs(os.path.join(COMFY, "custom_nodes"), exist_ok=True)
if os.path.isdir(os.path.join(mgr, ".git")):
    run("git pull --ff-only", cwd=mgr)
else:
    if os.path.isdir(mgr):
        shutil.rmtree(mgr)
    run(f"git clone --depth 1 https://github.com/ltdrdata/ComfyUI-Manager.git {mgr}")
mgr_req = os.path.join(mgr, "requirements.txt")
if os.path.isfile(mgr_req):
    run(f"{sys.executable} -m pip install -q -r {mgr_req}")
tick("manager", "ready")

# --- folders ---
for sub in (
    "checkpoints", "diffusion_models", "unet", "text_encoders", "clip", "clip_vision",
    "vae", "vae_approx", "loras", "latent_upscale_models", "upscale_models", "controlnet",
    "embeddings", "hypernetworks", "photomaker", "style_models", "animatediff_models",
    "ipadapter", "insightface", "inpaint", "facerestore_models", "sams", "detection",
):
    os.makedirs(os.path.join(COMFY, "models", sub), exist_ok=True)
os.makedirs("/content/hf_tmp", exist_ok=True)
open(os.path.join(COMFY, ".colab_ready"), "w").write(ver_used)

ui.set(status="READY", foot=f"finished in {elapsed()}")

In [ ]:
#@title 02  ·  Custom Nodes { display-mode: "form" }
git_urls = "https://github.com/GodL-x-SouL/LazyComfy.git" #@param {type:"string"}

import os, re, sys, time, shutil, subprocess, html
from IPython.display import display, HTML

COMFY = "/content/ComfyUI"
NODES = os.path.join(COMFY, "custom_nodes")

CSS = r"""
<style>
.om-wrap{width:98%;margin:0 auto;box-sizing:border-box;background:#141210;color:#e8dfd0;
border:1px solid #3f382c;padding:28px 32px 22px;font-family:Palatino,"Palatino Linotype","Book Antiqua",Georgia,serif;}
.om-top{display:flex;justify-content:space-between;align-items:baseline;gap:16px;}
.om-kicker{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.28em;color:#b8956c;text-transform:uppercase;}
.om-status{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.22em;text-transform:uppercase;}
.om-s-WORKING{color:#b8956c;} .om-s-READY,.om-s-LIVE{color:#8fa382;} .om-s-ERROR{color:#8a3a3a;} .om-s-STOPPED{color:#8a8070;}
.om-title{font-size:28px;font-weight:500;letter-spacing:.04em;color:#f3eadb;margin:6px 0 0;line-height:1.15;}
.om-rule{height:1px;background:#3f382c;margin:16px 0 18px;border:0;}
.om-row{display:flex;gap:18px;justify-content:space-between;padding:5px 0;font-size:16px;line-height:1.4;}
.om-k{color:#8a8070;min-width:180px;} .om-v{color:#e8dfd0;text-align:right;}
.om-foot{margin-top:16px;font-family:"Courier New",Courier,monospace;font-size:11px;color:#8a8070;letter-spacing:.06em;text-transform:uppercase;}
.om-note{color:#a89880;font-size:15px;line-height:1.5;}
.om-log{background:#0e0c0a;border:1px solid #2a261f;padding:12px 14px;max-height:220px;overflow:auto;
font-family:"Courier New",Courier,monospace;font-size:11px;line-height:1.55;color:#c8bba6;white-space:pre-wrap;word-break:break-word;}
</style>
"""

def esc(s):
    return html.escape(str(s) if s is not None else "")

class OMBox:
    def __init__(self, no, title):
        self.no, self.title, self.status, self.rows, self.foot, self.extra = no, title, "WORKING", [], "", ""
        self._h = None
        self.paint()
    def paint(self):
        rows = "".join(f'<div class="om-row"><span class="om-k">{esc(k)}</span><span class="om-v">{esc(v)}</span></div>' for k, v in self.rows)
        h = HTML(CSS + f'<div class="om-wrap"><div class="om-top"><div class="om-kicker">{esc(self.no)}</div><div class="om-status om-s-{esc(self.status)}">{esc(self.status)}</div></div><div class="om-title">{esc(self.title)}</div><div class="om-rule"></div>{rows}{self.extra}<div class="om-foot">{esc(self.foot)}</div></div>')
        self._h = display(h, display_id="om-02") if self._h is None else self._h.update(h) or self._h
    def set(self, **k):
        for a in ("rows", "status", "foot", "extra"):
            if a in k: setattr(self, a, k[a])
        self.paint()

def run(cmd, cwd=None):
    p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return p.returncode, p.stdout or "", p.stderr or ""

ui = OMBox("02", "Custom Nodes")
t0 = time.time()

if not os.path.isdir(NODES):
    ui.set(status="ERROR", extra='<div class="om-note">Run cell 01 first.</div>', foot="aborted")
    raise SystemExit

raw = (git_urls or "").strip()
urls = [u.strip().rstrip("/") for u in re.split(r"[\s,;]+", raw) if u.strip()]
urls = [u if u.endswith(".git") or "github.com" not in u else u + ".git" for u in urls]
urls = [u for u in urls if u.startswith("http")]

if not urls:
    ui.set(status="READY", extra='<div class="om-note">Paste one or more Git URLs above, then run this cell. Already-installed nodes are updated.</div>', foot="awaiting url")
    raise SystemExit

rows = []
errors = 0
changes = {}
for url in urls:
    name = url.split("/")[-1]
    if name.endswith(".git"):
        name = name[:-4]
    dest = os.path.join(NODES, name)
    action = "update" if os.path.isdir(os.path.join(dest, ".git")) else "install"
    rows.append((name, f"{action}…"))
    ui.set(rows=rows, foot=f"{action}  ·  {name}")

    if action == "update":
        _, old_head, _ = run("git rev-parse HEAD", cwd=dest)
        old_head = old_head.strip()
        c, o, e = run("git pull --ff-only", cwd=dest)
        if c != 0:
            c, o, e = run("git pull --rebase --autostash", cwd=dest)
        _, new_head, _ = run("git rev-parse HEAD", cwd=dest)
        new_head = new_head.strip()
        if c != 0:
            rows[-1] = (name, "update failed")
            errors += 1
            ui.set(rows=rows)
            continue
        if old_head == new_head:
            verb = "already up to date"
        else:
            _, short, _ = run(f"git diff --shortstat {old_head} {new_head}", cwd=dest)
            _, stat, _ = run(f"git diff --stat {old_head} {new_head}", cwd=dest)
            changes[name] = (short.strip(), stat.strip())
            verb = short.strip() or "updated"
    else:
        if os.path.isdir(dest):
            shutil.rmtree(dest)
        c, o, e = run(f"git clone --recursive {url} {dest}")
        if c != 0:
            rows[-1] = (name, "failed to clone")
            errors += 1
            ui.set(rows=rows)
            continue
        verb = "installed"
        _, istat, _ = run("git show --stat --oneline HEAD", cwd=dest)
        changes[name] = ("new clone", "\n".join(istat.splitlines()[:15]).strip())

    for req in ("requirements.txt", "requirements-colab.txt"):
        rp = os.path.join(dest, req)
        if os.path.isfile(rp):
            run(f"{sys.executable} -m pip install -q -r {rp}")
    if os.path.isfile(os.path.join(dest, "install.py")):
        run(f"{sys.executable} install.py", cwd=dest)

    rows[-1] = (name, verb)
    ui.set(rows=rows)

sec = int(time.time() - t0)
log_txt = ""
for n, (s, st) in changes.items():
    log_txt += f"{n}  ·  {s}\n{st}\n\n"
extra = f'<div class="om-log">{esc(log_txt.strip())}</div>' if log_txt.strip() else ""
ui.set(status="ERROR" if errors else "READY", extra=extra,
       foot=("restart the server in cell 04 to load new nodes  ·  " if rows else "") + f"finished in {sec}s")

In [ ]:
#@title 03  ·  Models { display-mode: "form" }
diffusion_models = "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/blob/main/diffusion_models/qwen_image_2.1_int8_convrot.safetensors" #@param {type:"string"}
unet = "" #@param {type:"string"}
text_encoders = "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/blob/main/text_encoders/qwen3vl_8b_w4a8.safetensors" #@param {type:"string"}
vae = "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/blob/main/vae/qwen_image_2.1_vae_bf16.safetensors" #@param {type:"string"}
loras = "" #@param {type:"string"}
latent_upscale_models = "" #@param {type:"string"}
custom_path = "" #@param {type:"string"}
custom_url = "" #@param {type:"string"}
hf_token = "" #@param {type:"string"}
skip_existing = True #@param {type:"boolean"}

import os, warnings
warnings.filterwarnings("ignore")
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_EXPERIMENTAL_WARNING"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["PYTHONWARNINGS"] = "ignore"

import re, sys, time, shutil, threading, html
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import unquote, urlparse
from IPython.display import display, HTML

COMFY = "/content/ComfyUI"
TOKEN = (hf_token or os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or "").strip()

MAP = [
    (diffusion_models, os.path.join(COMFY, "models", "diffusion_models")),
    (unet, os.path.join(COMFY, "models", "unet")),
    (text_encoders, os.path.join(COMFY, "models", "text_encoders")),
    (vae, os.path.join(COMFY, "models", "vae")),
    (loras, os.path.join(COMFY, "models", "loras")),
    (latent_upscale_models, os.path.join(COMFY, "models", "latent_upscale_models")),
]

CSS = r"""
<style>
.om-wrap{width:98%;margin:0 auto;box-sizing:border-box;background:#141210;color:#e8dfd0;
border:1px solid #3f382c;padding:28px 32px 22px;font-family:Palatino,"Palatino Linotype","Book Antiqua",Georgia,serif;}
.om-top{display:flex;justify-content:space-between;align-items:baseline;gap:16px;}
.om-kicker{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.28em;color:#b8956c;text-transform:uppercase;}
.om-status{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.22em;text-transform:uppercase;}
.om-s-WORKING{color:#b8956c;} .om-s-READY,.om-s-LIVE{color:#8fa382;} .om-s-ERROR{color:#8a3a3a;} .om-s-STOPPED{color:#8a8070;}
.om-title{font-size:28px;font-weight:500;letter-spacing:.04em;color:#f3eadb;margin:6px 0 0;line-height:1.15;}
.om-rule{height:1px;background:#3f382c;margin:16px 0 18px;border:0;}
.om-name{font-size:18px;color:#f3eadb;} .om-sub{color:#8a8070;font-size:14px;margin-top:2px;}
.om-bar-track{height:2px;background:#2a261f;margin:16px 0 10px;} .om-bar-fill{height:2px;background:#b8956c;}
.om-stat{font-family:"Courier New",Courier,monospace;font-size:11px;color:#a89880;letter-spacing:.04em;}
.om-hist{margin-top:16px;} .om-hrow{display:flex;justify-content:space-between;gap:16px;padding:3px 0;font-size:15px;}
.om-hk{color:#8a8070;} .om-hv{color:#c8bba6;}
.om-foot{margin-top:16px;font-family:"Courier New",Courier,monospace;font-size:11px;color:#8a8070;letter-spacing:.06em;text-transform:uppercase;}
.om-note{color:#a89880;font-size:15px;line-height:1.5;}
</style>
"""

def esc(s):
    return html.escape(str(s) if s is not None else "")

def fmt_bytes(n):
    n = float(n or 0)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or u == "TB":
            return f"{n:.0f} {u}" if u == "B" else f"{n:.2f} {u}"
        n /= 1024.0

def fmt_eta(sec):
    if sec is None or sec == float("inf") or sec < 0:
        return "—"
    sec = int(sec)
    if sec < 60: return f"{sec}s"
    if sec < 3600: return f"{sec//60}m {sec%60:02d}s"
    return f"{sec//3600}h {(sec%3600)//60:02d}m"

class OMBox:
    def __init__(self):
        self.status, self.foot = "WORKING", ""
        self.file = self.folder = ""
        self.done = self.total = 0
        self.speed = 0.0
        self.eta = None
        self.hist = []
        self.queue_left = 0
        self._h = None
        self.paint()
    def paint(self):
        pct = 0 if not self.total else min(100.0, 100.0 * self.done / self.total)
        hist = "".join(f'<div class="om-hrow"><span class="om-hk">{esc(a)}</span><span class="om-hv">{esc(b)}</span></div>' for a, b in self.hist[-12:])
        extra = ""
        if self.file:
            extra = f"""
            <div class="om-name">{esc(self.file)}</div>
            <div class="om-sub">{esc(self.folder)}{('  ·  ' + str(self.queue_left) + ' queued') if self.queue_left else ''}</div>
            <div class="om-bar-track"><div class="om-bar-fill" style="width:{pct:.2f}%"></div></div>
            <div class="om-stat">{fmt_bytes(self.done)} / {fmt_bytes(self.total) if self.total else '—'}  ·  {fmt_bytes(self.speed)}/s  ·  {fmt_eta(self.eta)} remaining  ·  {pct:.1f}%</div>
            """
        if self.hist:
            extra += f'<div class="om-hist">{hist}</div>'
        if not self.file and not self.hist:
            extra = '<div class="om-note">Paste Hugging Face, Civitai, or direct file URLs above.</div>'
        h = HTML(CSS + f"""
        <div class="om-wrap">
          <div class="om-top"><div class="om-kicker">03</div>
            <div class="om-status om-s-{esc(self.status)}">{esc(self.status)}</div></div>
          <div class="om-title">Models</div>
          <div class="om-rule"></div>
          {extra}
          <div class="om-foot">{esc(self.foot)}</div>
        </div>""")
        self._h = display(h, display_id="om-03") if self._h is None else self._h.update(h) or self._h

ui = OMBox()

def split_urls(s):
    if not s or not str(s).strip():
        return []
    return [p.strip() for p in re.split(r"[\s,;]+", str(s).strip()) if p.strip().startswith(("http://", "https://", "hf://"))]

def parse_hf(url):
    url = url.strip()
    if url.startswith("hf://"):
        rest = url[5:]
        parts = rest.split("/", 2)
        if len(parts) >= 3:
            return {"repo_id": f"{parts[0]}/{parts[1]}", "revision": "main", "filename": unquote(parts[2].split("?")[0])}
        return None
    m = re.match(r"https?://huggingface\.co/([^/]+)/([^/]+)/(?:blob|resolve|raw)/([^/]+)/(.*)$", url)
    if m:
        org, repo, rev, path = m.groups()
        return {"repo_id": f"{org}/{repo}", "revision": rev, "filename": unquote(path.split("?")[0])}
    return None

def headers(extra=None):
    h = {"User-Agent": "Mozilla/5.0"}
    if TOKEN:
        h["Authorization"] = f"Bearer {TOKEN}"
    if extra:
        h.update(extra)
    return h

def resolve_job(url, dest_dir):
    import requests
    meta = parse_hf(url)
    hdrs = headers()
    if meta:
        name = os.path.basename(meta["filename"]) or "download.bin"
        file_url = f"https://huggingface.co/{meta['repo_id']}/resolve/{meta['revision']}/{meta['filename']}"
        r = requests.head(file_url, headers=hdrs, allow_redirects=True, timeout=30)
        if r.status_code >= 400:
            r = requests.get(file_url, headers=hdrs, stream=True, timeout=30)
            final, size = r.url, int(r.headers.get("Content-Length") or 0)
            r.close()
        else:
            final, size = r.url, int(r.headers.get("Content-Length") or 0)
        return {"url": url, "final": final, "name": name, "size": size, "dest": os.path.join(dest_dir, name), "dir": dest_dir, "headers": hdrs}

    r = requests.head(url, headers=hdrs, allow_redirects=True, timeout=30)
    final, size = r.url, int(r.headers.get("Content-Length") or 0)
    cd = r.headers.get("Content-Disposition") or ""
    m = re.search(r'filename\*?=(?:UTF-8\'\')?"?([^";]+)"?', cd, re.I)
    name = os.path.basename(unquote(m.group(1))) if m else (os.path.basename(urlparse(final).path) or "download.bin")
    if not size or r.status_code >= 400:
        r = requests.get(url, headers=hdrs, stream=True, timeout=30)
        final, size = r.url, int(r.headers.get("Content-Length") or 0)
        cd = r.headers.get("Content-Disposition") or ""
        m = re.search(r'filename\*?=(?:UTF-8\'\')?"?([^";]+)"?', cd, re.I)
        if m:
            name = os.path.basename(unquote(m.group(1)))
        r.close()
    name = name or "download.bin"
    return {"url": url, "final": final, "name": name, "size": size, "dest": os.path.join(dest_dir, name), "dir": dest_dir, "headers": hdrs}

def download_stream(job, progress):
    import requests
    dest, url, hdrs = job["dest"], job["final"], job["headers"]
    tmp = dest + ".part"
    with requests.get(url, headers=hdrs, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length") or job["size"] or 0)
        done = 0
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1024 * 256):
                if not chunk:
                    continue
                f.write(chunk)
                done += len(chunk)
                progress(done, total)
    os.replace(tmp, dest)
    progress(os.path.getsize(dest), os.path.getsize(dest))

def download_parallel(job, progress, n=8):
    import requests
    dest, url, hdrs, size = job["dest"], job["final"], job["headers"], job["size"]
    if not size or size < 8 * 1024 * 1024:
        return download_stream(job, progress)
    probe = requests.head(url, headers=hdrs, allow_redirects=True, timeout=30)
    if probe.headers.get("Accept-Ranges", "").lower() != "bytes":
        return download_stream(job, progress)
    tmp = dest + ".part"
    with open(tmp, "wb") as f:
        f.truncate(size)
    lock = threading.Lock()
    done = [0]
    err = [None]
    chunk = (size + n - 1) // n

    def worker(i):
        start = i * chunk
        end = min(start + chunk, size) - 1
        if start > end:
            return
        h = dict(hdrs)
        h["Range"] = f"bytes={start}-{end}"
        for attempt in range(4):
            try:
                with requests.get(url, headers=h, stream=True, timeout=60) as r:
                    r.raise_for_status()
                    with open(tmp, "r+b") as f:
                        f.seek(start)
                        for part in r.iter_content(1024 * 256):
                            if not part:
                                continue
                            f.write(part)
                            with lock:
                                done[0] += len(part)
                                progress(done[0], size)
                return
            except Exception as e:
                err[0] = e
                time.sleep(1.5 * (attempt + 1))
        raise err[0]

    with ThreadPoolExecutor(max_workers=n) as ex:
        futs = [ex.submit(worker, i) for i in range(n)]
        for fut in futs:
            fut.result()
    os.replace(tmp, dest)

jobs = []
for blob, folder in MAP:
    for u in split_urls(blob):
        jobs.append((u, folder))
if (custom_url or "").strip() and (custom_path or "").strip():
    for u in split_urls(custom_url):
        jobs.append((u, custom_path.strip()))

if not jobs:
    ui.status, ui.foot = "READY", "awaiting url"
    ui.paint()
elif not os.path.isdir(COMFY):
    ui.status, ui.foot = "ERROR", "run cell 01 first"
    ui.paint()
else:
    state = {"done": 0, "total": 0, "t0": time.time(), "ema": 0.0, "last": 0, "lt": time.time()}
    lock = threading.Lock()

    def on_progress(done, total):
        now = time.time()
        with lock:
            state["done"] = done
            state["total"] = total or 0
            dt = now - state["lt"]
            if dt >= 0.15:
                inst = (done - state["last"]) / dt if dt > 0 else 0
                state["ema"] = inst if state["ema"] == 0 else (0.25 * inst + 0.75 * state["ema"])
                state["last"] = done
                state["lt"] = now

    t0 = time.time()
    failed = 0
    for i, (url, folder) in enumerate(jobs):
        os.makedirs(folder, exist_ok=True)
        ui.queue_left = len(jobs) - i - 1
        ui.folder = folder
        ui.file = "resolving"
        ui.done = ui.total = 0
        ui.speed = 0
        ui.eta = None
        ui.foot = f"file {i+1} of {len(jobs)}"
        ui.paint()
        try:
            job = resolve_job(url, folder)
        except Exception as e:
            ui.hist.append((url, f"resolve failed  ·  {e}"))
            failed += 1
            ui.paint()
            continue

        ui.file = job["name"]
        ui.total = job["size"]
        ui.paint()

        if skip_existing and os.path.isfile(job["dest"]):
            existing = os.path.getsize(job["dest"])
            if not job["size"] or existing == job["size"]:
                ui.hist.append((job["name"], "already present"))
                ui.paint()
                continue

        state.update({"done": 0, "total": job["size"], "t0": time.time(), "ema": 0.0, "last": 0, "lt": time.time()})
        err_box = [None]

        def work():
            try:
                download_parallel(job, on_progress)
            except Exception as e:
                err_box[0] = e

        th = threading.Thread(target=work, daemon=True)
        th.start()
        while th.is_alive():
            with lock:
                ui.done, ui.total = state["done"], state["total"]
                ui.speed = state["ema"]
                ui.eta = (ui.total - ui.done) / ui.speed if ui.speed > 1 else None
            ui.paint()
            time.sleep(0.12)
        th.join()
        if err_box[0]:
            if os.path.isfile(job["dest"] + ".part"):
                try: os.remove(job["dest"] + ".part")
                except OSError: pass
            ui.hist.append((job["name"], f"failed  ·  {err_box[0]}"))
            failed += 1
        else:
            ui.hist.append((job["name"], f"{fmt_bytes(os.path.getsize(job['dest']))}  →  {folder}"))
        ui.paint()

    ui.file = ""
    ui.status = "ERROR" if failed else "READY"
    ui.foot = f"{len(jobs) - failed} of {len(jobs)} placed  ·  {int(time.time()-t0)}s"
    ui.paint()

In [ ]:
#@title 04  ·  Launch { display-mode: "form" }
port = 8188 #@param {type:"integer"}

import os, re, sys, time, json, signal, socket, threading, subprocess, html, configparser, warnings
warnings.filterwarnings("ignore")

from collections import deque
from IPython.display import display, HTML, Javascript

COMFY = "/content/ComfyUI"
PORT = int(port or 8188)
CF = "/usr/local/bin/cloudflared"
LOGS = deque(maxlen=400)
LOG_SEQ = [0]
PROCS = []
def log_add(msg):
    log_add(msg)
    LOG_SEQ[0] += 1
MID = f"om04_{int(time.time() * 1000)}"

CSS = f"""
<style>
#{MID} {{
  width:98%;margin:0 auto;box-sizing:border-box;background:#141210;color:#e8dfd0;
  border:1px solid #3f382c;padding:28px 32px 22px;
  font-family:Palatino,"Palatino Linotype","Book Antiqua",Georgia,serif;
}}
#{MID} .om-top{{display:flex;justify-content:space-between;align-items:baseline;gap:16px;}}
#{MID} .om-kicker{{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.28em;color:#b8956c;text-transform:uppercase;}}
#{MID} .om-status{{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.22em;text-transform:uppercase;}}
#{MID} .om-s-WORKING{{color:#b8956c;}} #{MID} .om-s-READY,#{MID} .om-s-LIVE{{color:#8fa382;}}
#{MID} .om-s-ERROR{{color:#8a3a3a;}} #{MID} .om-s-STOPPED{{color:#8a8070;}}
#{MID} .om-title{{font-size:28px;font-weight:500;letter-spacing:.04em;color:#f3eadb;margin:6px 0 0;line-height:1.15;}}
#{MID} .om-rule{{height:1px;background:#3f382c;margin:16px 0 18px;border:0;}}
#{MID} a.om-url{{display:none;font-size:20px;color:#f3eadb;word-break:break-all;line-height:1.35;
  text-decoration:none;border-bottom:1px solid #5a4e3c;cursor:pointer;user-select:text;width:fit-content;max-width:100%;}}
#{MID} a.om-url:hover{{color:#fff8ee;border-bottom-color:#b8956c;}}
#{MID} .om-wait{{color:#a89880;font-size:15px;line-height:1.5;}}
#{MID} .om-sub{{margin-top:8px;color:#8a8070;font-size:15px;}}
#{MID} .om-loglab{{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.22em;text-transform:uppercase;color:#b8956c;margin:18px 0 8px;}}
#{MID} .om-log{{background:#0e0c0a;border:1px solid #2a261f;padding:12px 14px;height:340px;max-height:340px;overflow-y:auto;overflow-x:hidden;
  font-family:"Courier New",Courier,monospace;font-size:11px;line-height:1.55;color:#c8bba6;white-space:pre-wrap;word-break:break-word;
  overscroll-behavior:contain;overflow-anchor:none;}}
#{MID} .om-foot{{margin-top:16px;font-family:"Courier New",Courier,monospace;font-size:11px;color:#8a8070;letter-spacing:.06em;text-transform:uppercase;}}
</style>
"""

class OMBox:
    def __init__(self):
        self.status, self.url, self.sub, self.foot = "WORKING", "", "starting", ""
        self._seq = 0
        self._sig = None
        display(HTML(CSS + f"""
        <div id="{MID}">
          <div class="om-top"><div class="om-kicker">04</div>
            <div class="om-status om-s-WORKING" data-k="status">WORKING</div></div>
          <div class="om-title">Server</div>
          <div class="om-rule"></div>
          <a class="om-url" data-k="url" href="about:blank" target="_blank" rel="noopener noreferrer" referrerpolicy="no-referrer"></a>
          <div class="om-wait" data-k="wait">waiting for a public url</div>
          <div class="om-sub" data-k="sub">starting</div>
          <div class="om-loglab">Log</div>
          <div class="om-log" data-k="log"></div>
          <div class="om-foot" data-k="foot"></div>
        </div>
        <script>
        (function(){{
          var root = document.getElementById("{MID}");
          if (!root) return;
          var log = root.querySelector('[data-k=log]');
          var follow = true;
          log.addEventListener('scroll', function(){{
            follow = (log.scrollHeight - log.scrollTop - log.clientHeight) < 48;
          }});
          var a = root.querySelector('[data-k=url]');
          a.addEventListener('click', function(ev){{
            var href = a.getAttribute('href');
            if (!href || href === 'about:blank') return;
            ev.preventDefault();
            window.open(href, '_blank', 'noopener,noreferrer');
          }});
          window.__om04Apply_{MID} = function(d){{
            var st = root.querySelector('[data-k=status]');
            st.textContent = d.status;
            st.className = 'om-status om-s-' + d.status;
            var w = root.querySelector('[data-k=wait]');
            if (d.url) {{
              a.href = d.url;
              a.textContent = d.url;
              a.style.display = 'inline';
              w.style.display = 'none';
            }} else {{
              a.style.display = 'none';
              a.removeAttribute('href');
              a.textContent = '';
              w.style.display = 'block';
            }}
            root.querySelector('[data-k=sub]').textContent = d.sub || '';
            root.querySelector('[data-k=foot]').textContent = d.foot || '';
            if (log.textContent !== d.log) {{
              log.textContent = d.log;
              if (follow) requestAnimationFrame(function(){{ log.scrollTop = log.scrollHeight; }});
            }}
          }};
        }})();
        </script>
        """), display_id=MID)
        display(Javascript("void 0"), display_id=MID + "_js")
        self.paint(force=True)

    def paint(self, force=False):
        payload = {
            "status": self.status,
            "url": self.url,
            "sub": self.sub,
            "foot": self.foot,
            "log": "\n".join(LOGS),
        }
        sig = (payload["status"], payload["url"], payload["sub"], payload["foot"], payload["log"])
        if not force and sig == self._sig:
            return
        self._sig = sig
        self._seq += 1
        code = f"window.__om04Apply_{MID} && window.__om04Apply_{MID}({json.dumps(payload)}); /*{self._seq}*/"
        display(Javascript(code), display_id=MID + "_js", update=True)

ui = OMBox()

def pump(proc, tag):
    try:
        for line in iter(proc.stdout.readline, ""):
            if not line:
                break
            log_add(f"{tag}  {line.rstrip()}")
    except Exception:
        pass

def free_port(p):
    subprocess.run(f"fuser -k {p}/tcp", shell=True, capture_output=True)
    subprocess.run(f"lsof -ti:{p} | xargs -r kill -9", shell=True, capture_output=True)

def kill_tree(proc):
    if not proc or proc.poll() is not None:
        return
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
    except Exception:
        try: proc.terminate()
        except Exception: pass
    try:
        proc.wait(timeout=4)
    except Exception:
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except Exception:
            try: proc.kill()
            except Exception: pass

def cleanup():
    for p in list(PROCS):
        kill_tree(p)
    PROCS.clear()
    subprocess.run("pkill -f cloudflared", shell=True, capture_output=True)
    subprocess.run("pkill -f 'ComfyUI/main.py'", shell=True, capture_output=True)
    free_port(PORT)

def port_open(p):
    s = socket.socket(); s.settimeout(1.0)
    try:
        s.connect(("127.0.0.1", p)); s.close(); return True
    except Exception:
        return False

def http_ready(p):
    import urllib.request
    for path in ("/system_stats", "/", "/queue"):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{p}{path}", timeout=2)
            return True
        except Exception:
            continue
    return False

def help_text():
    p = subprocess.run([sys.executable, "main.py", "--help"], cwd=COMFY,
                       capture_output=True, text=True, timeout=40)
    return (p.stdout or "") + "\n" + (p.stderr or "")

def supported(help_blob, flag):
    return bool(re.search(rf"(?:^|\s){re.escape(flag)}(?:\s|$)", help_blob))

def gpu_mb():
    try:
        o = subprocess.run(
            "nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits",
            shell=True, capture_output=True, text=True,
        ).stdout.strip()
        return int(o.splitlines()[0].split(",")[0])
    except Exception:
        return 0

def relax_remote_security():
    os.environ["COMFYUI_MANAGER_SECURITY_LEVEL"] = "weak"
    paths = [
        os.path.join(COMFY, "user", "default", "ComfyUI-Manager", "config.ini"),
        os.path.join(COMFY, "user", "__manager", "config.ini"),
        os.path.join(COMFY, "custom_nodes", "ComfyUI-Manager", "config.ini"),
    ]
    for path in paths:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        cfg = configparser.ConfigParser()
        if os.path.isfile(path):
            try: cfg.read(path)
            except Exception: cfg = configparser.ConfigParser()
        if "default" not in cfg:
            cfg["default"] = {}
        cfg["default"]["security_level"] = "weak"
        with open(path, "w") as f:
            cfg.write(f)
    log_add("manager security  weak")

def build_cmd():
    help_blob = help_text()
    log_add("read main.py --help")
    cmd = [sys.executable, "-u", "main.py"]
    for flag, val in (("--listen", "127.0.0.1"), ("--port", str(PORT)), ("--preview-method", "auto")):
        if supported(help_blob, flag):
            cmd += [flag, val]
    for flag in ("--disable-auto-launch", "--disable-metadata", "--enable-cors-header"):
        if supported(help_blob, flag):
            cmd.append(flag)
    mb = gpu_mb()
    choice = None
    if mb and mb < 15000 and supported(help_blob, "--lowvram"):
        choice = "--lowvram"
    elif mb >= 40000 and supported(help_blob, "--highvram"):
        choice = "--highvram"
    if choice:
        cmd.append(choice)
        log_add(f"vram flag  {choice}  ·  {mb} MB")
    else:
        log_add(f"vram flag  default  ·  {mb or 'unknown'} MB")
    return cmd

def blob():
    return "\n".join(LOGS)

def find_url():
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", blob())
    return m.group(0) if m else None

def tunnel_ready():
    return bool(re.search(
        r"Connection registered|Registered tunnel connection|Registered conn|connIndex\s*=\s*\d+",
        blob(), re.I,
    ))

def tunnel_auth_fail():
    return bool(re.search(
        r"unauthorized|no authorization|login required|failed to request a quick|couldn't request a tunnel|429|Too many",
        blob(), re.I,
    ))

def wait_on_logs(seconds, pred):
    deadline = time.time() + seconds
    last = -1
    while time.time() < deadline:
        if pred():
            return True
        if LOG_SEQ[0] != last:
            last = LOG_SEQ[0]
            ui.paint()
        time.sleep(0.35)
    return False

def ensure_cloudflared():
    if os.path.isfile(CF) and os.access(CF, os.X_OK):
        return
    subprocess.run(
        "wget -q -O /usr/local/bin/cloudflared "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        shell=True,
    )
    subprocess.run("chmod +x /usr/local/bin/cloudflared", shell=True)

def start_cloudflared(extra):
    args = [CF, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"] + extra
    log_add("tunnel  " + " ".join(args[1:]))
    proc = subprocess.Popen(
        args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True,
    )
    PROCS.append(proc)
    threading.Thread(target=pump, args=(proc, "tunnel"), daemon=True).start()
    return proc

def open_cloudflare():
    attempts = [
        ["--protocol", "http2"],
        ["--protocol", "http2", "--edge-ip-version", "4"],
        [],
    ]
    for i, extra in enumerate(attempts, 1):
        log_add(f"cloudflare  attempt {i}/{len(attempts)}")
        ui.paint()
        proc = start_cloudflared(extra)
        wait_on_logs(80, lambda: (find_url() and tunnel_ready()) or tunnel_auth_fail() or proc.poll() is not None)
        url = find_url()
        if url and (tunnel_ready() or wait_on_logs(20, tunnel_ready)):
            log_add("cloudflare  edge connection registered")
            return proc, url
        if url and proc.poll() is None and not tunnel_auth_fail():
            log_add("cloudflare  url printed  ·  waiting briefly for edge")
            time.sleep(4)
            if proc.poll() is None:
                return proc, url
        reason = "authorization failed" if tunnel_auth_fail() else "no registered connection"
        log_add(f"cloudflare  attempt {i} failed  ·  {reason}")
        kill_tree(proc)
        if proc in PROCS:
            PROCS.remove(proc)
        time.sleep(2.0)
    return None, None

def fail(msg):
    ui.status, ui.sub = "ERROR", msg
    ui.paint(force=True)
    cleanup()

if not os.path.isfile(os.path.join(COMFY, "main.py")):
    ui.status, ui.sub, ui.foot = "ERROR", "ComfyUI is not installed", "run cell 01 first"
    ui.paint(force=True)
else:
    ensure_cloudflared()
    cleanup()
    time.sleep(0.6)
    relax_remote_security()
    log_add("ports cleared")
    ui.foot = "interrupt this cell to stop the server and the tunnel"
    ui.paint(force=True)

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    cmd = build_cmd()
    log_add("exec  " + " ".join(cmd))
    comfy = subprocess.Popen(
        cmd, cwd=COMFY, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True, env=env,
    )
    PROCS.append(comfy)
    threading.Thread(target=pump, args=(comfy, "comfy"), daemon=True).start()

    ui.sub = f"waiting for 127.0.0.1:{PORT}"
    ui.paint(force=True)

    ready = wait_on_logs(240, lambda: comfy.poll() is not None or (port_open(PORT) and http_ready(PORT)))
    if comfy.poll() is not None:
        fail("comfyui exited before it became ready")
        ready = False
    elif not (port_open(PORT) and http_ready(PORT)):
        fail("timed out waiting for the local server")
        ready = False
    else:
        ready = True

    if ready:
        log_add(f"server is accepting connections on {PORT}")
        ui.sub = "server up  ·  opening cloudflare tunnel"
        ui.paint(force=True)

        tunnel, public = open_cloudflare()
        if not public:
            fail("cloudflare did not publish a url")
        else:
            ui.status = "LIVE"
            ui.url = public
            ui.sub = f"{PORT}  comfyui up   ·   cloudflare up   ·   click the url to open in a new tab"
            ui.foot = "interrupt this cell to stop the server and the tunnel"
            ui.paint(force=True)
            last = -1
            try:
                while True:
                    if comfy.poll() is not None:
                        ui.status, ui.sub = "ERROR", "comfyui process ended"
                        ui.paint(force=True)
                        break
                    if tunnel.poll() is not None:
                        ui.status, ui.sub = "ERROR", "tunnel process ended"
                        ui.paint(force=True)
                        break
                    if LOG_SEQ[0] != last:
                        last = LOG_SEQ[0]
                        ui.paint()
                    time.sleep(0.4)
            except KeyboardInterrupt:
                log_add("interrupt received")
            finally:
                cleanup()
                ui.status = "STOPPED"
                ui.url = ""
                ui.sub = f"server and tunnel stopped  ·  port {PORT} released"
                ui.foot = "run this cell again to start"
                ui.paint(force=True)

In [ ]:
#@title 05  ·  Cleanup { display-mode: "form" }
confirm = False #@param {type:"boolean"}

import os, time, shutil, html
from IPython.display import display, HTML

COMFY = "/content/ComfyUI"
TARGETS = (
    os.path.join(COMFY, "input"),
    os.path.join(COMFY, "output"),
)

CSS = r"""
<style>
.om-wrap{width:98%;margin:0 auto;box-sizing:border-box;background:#141210;color:#e8dfd0;
border:1px solid #3f382c;padding:28px 32px 22px;font-family:Palatino,"Palatino Linotype","Book Antiqua",Georgia,serif;}
.om-top{display:flex;justify-content:space-between;align-items:baseline;gap:16px;}
.om-kicker{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.28em;color:#b8956c;text-transform:uppercase;}
.om-status{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.22em;text-transform:uppercase;}
.om-s-WORKING{color:#b8956c;} .om-s-READY,.om-s-LIVE{color:#8fa382;} .om-s-ERROR{color:#8a3a3a;} .om-s-STOPPED{color:#8a8070;}
.om-title{font-size:28px;font-weight:500;letter-spacing:.04em;color:#f3eadb;margin:6px 0 0;line-height:1.15;}
.om-rule{height:1px;background:#3f382c;margin:16px 0 18px;border:0;}
.om-row{display:flex;gap:18px;justify-content:space-between;padding:5px 0;font-size:16px;line-height:1.4;}
.om-k{color:#8a8070;min-width:180px;} .om-v{color:#e8dfd0;text-align:right;}
.om-foot{margin-top:16px;font-family:"Courier New",Courier,monospace;font-size:11px;color:#8a8070;letter-spacing:.06em;text-transform:uppercase;}
.om-note{color:#a89880;font-size:15px;line-height:1.5;}
</style>
"""

def esc(s):
    return html.escape(str(s) if s is not None else "")

def fmt_bytes(n):
    n = float(n or 0)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or u == "TB":
            return f"{n:.0f} {u}" if u == "B" else f"{n:.2f} {u}"
        n /= 1024.0

class OMBox:
    def __init__(self, no, title):
        self.no, self.title, self.status, self.rows, self.foot, self.extra = no, title, "WORKING", [], "", ""
        self._h = None
        self.paint()
    def paint(self):
        rows = "".join(f'<div class="om-row"><span class="om-k">{esc(k)}</span><span class="om-v">{esc(v)}</span></div>' for k, v in self.rows)
        h = HTML(CSS + f'<div class="om-wrap"><div class="om-top"><div class="om-kicker">{esc(self.no)}</div><div class="om-status om-s-{esc(self.status)}">{esc(self.status)}</div></div><div class="om-title">{esc(self.title)}</div><div class="om-rule"></div>{rows}{self.extra}<div class="om-foot">{esc(self.foot)}</div></div>')
        self._h = display(h, display_id="om-05") if self._h is None else self._h.update(h) or self._h
    def set(self, **k):
        for a in ("rows", "status", "foot", "extra"):
            if a in k: setattr(self, a, k[a])
        self.paint()

ui = OMBox("05", "Cleanup")
t0 = time.time()

if not confirm:
    ui.set(status="READY", extra='<div class="om-note">Tick confirm above, then run this cell. This deletes every file inside input/ and output/ (the folders themselves are kept).</div>', foot="awaiting confirmation")
    raise SystemExit

rows = []
for folder in TARGETS:
    label = os.path.relpath(folder, COMFY)
    if not os.path.isdir(folder):
        rows.append((label, "folder missing"))
        continue
    n_files, n_dirs, freed = 0, 0, 0
    for entry in os.listdir(folder):
        p = os.path.join(folder, entry)
        try:
            if os.path.isfile(p) or os.path.islink(p):
                freed += os.path.getsize(p)
                os.remove(p)
                n_files += 1
            elif os.path.isdir(p):
                for root, dirs, files in os.walk(p):
                    for fn in files:
                        fp = os.path.join(root, fn)
                        try: freed += os.path.getsize(fp)
                        except OSError: pass
                        n_files += 1
                n_dirs += 1 + sum(1 for _, d, _ in os.walk(p) for _ in d)
                shutil.rmtree(p)
        except OSError as e:
            rows.append((label, f"error  ·  {e}"))
            break
    else:
        rows.append((label, f"{n_files} files cleared  ·  {fmt_bytes(freed)} freed" if n_files else "already empty"))
    ui.set(rows=rows)

sec = int(time.time() - t0)
ui.set(status="READY", foot=f"finished in {sec}s")
